# Jupyter + PyVista 四个 OpenDX 三维可视化案例

本 notebook 是 Week 13 的主线案例。它使用本目录下的 OpenDX 样例数据，通过 `opendx_to_pyvista.py` 转换为 PyVista/VTK 数据对象，并在 Jupyter 中完成三维交互展示和截图导出。

四个案例：

1. Colorado 地形照片 + 高程贴图
2. 水分子电子密度等值面
3. 气象模拟：云水标量场 + 风场 glyph
4. MRI 医学体数据切片

## 0. 环境准备

建议从 `week13` 目录启动：

```bash
conda activate ai4math-vis
jupyter lab opendx_jupyter_pyvista_cases.ipynb
```

如果从仓库根目录启动，本 notebook 也会自动寻找 `week13/opendx_to_pyvista.py`。

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pyvista as pv
import ipywidgets as widgets

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "opendx_to_pyvista.py").exists() and (NOTEBOOK_DIR / "week13" / "opendx_to_pyvista.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "week13"

sys.path.insert(0, str(NOTEBOOK_DIR.resolve()))

from opendx_to_pyvista import (
    DATA_DIR,
    load_colorado_terrain,
    load_mri,
    load_storm_cloud_and_wind,
    load_watermolecule,
)

DATA_DIR = NOTEBOOK_DIR / "opendx_data"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "opendx_cases"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pv.global_theme.background = "white"
pv.global_theme.font.color = "black"

print("PyVista", pv.__version__)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

In [ ]:
try:
    pv.set_jupyter_backend("trame")
    print("Jupyter backend: trame")
except Exception as exc:
    print("Could not enable trame backend:", exc)
    print("The off-screen screenshot cells can still be used.")

## 动态交互方式

每个案例都使用 `ipywidgets.interact` 提供参数控件。为了避免三维场景在拖动过程中频繁重建，滑块默认在松开鼠标后才更新。三维视图本身仍可在 Jupyter 中旋转、缩放和平移。

## 1. Colorado 地形照片 + 高程贴图

数据文件：

- `colorado.tiff`：400 x 400 RGB 地形影像
- `colorado_elev.vit`：400 x 400 高程数据
- `colo_elev.general`：OpenDX general importer 头文件

这个案例展示二维图像如何贴到三维高程曲面上。

In [ ]:
terrain, colorado_texture = load_colorado_terrain(DATA_DIR, z_scale=2.0)
terrain

In [ ]:
def show_colorado_terrain(z_scale=2.0):
    terrain_dynamic, texture_dynamic = load_colorado_terrain(DATA_DIR, z_scale=z_scale)
    p = pv.Plotter(window_size=(900, 650))
    p.add_mesh(terrain_dynamic, texture=texture_dynamic)
    p.add_text(f"Colorado terrain, z scale = {z_scale:.1f}", position="upper_left", font_size=12)
    p.add_axes(line_width=2)
    p.camera_position = [(560, -620, 430), (200, 200, 220), (0, 0, 1)]
    return p.show()

widgets.interact(
    show_colorado_terrain,
    z_scale=widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.5, description="height", continuous_update=False),
);

In [ ]:
terrain_png = OUTPUT_DIR / "case1_colorado_terrain.png"
p = pv.Plotter(off_screen=True, window_size=(1200, 850))
p.add_mesh(terrain, texture=colorado_texture)
p.add_text("Colorado terrain", position="upper_left", font_size=12)
p.add_axes(line_width=2)
p.camera_position = [(560, -620, 430), (200, 200, 220), (0, 0, 1)]
p.screenshot(terrain_png)
p.close()
terrain_png

## 2. 水分子电子密度等值面

数据文件：`watermolecule.dx`

这是一个 40 x 60 x 20 的三维标量场。等值面可以显示电子密度较高的空间区域。

In [ ]:
water = load_watermolecule(DATA_DIR)
density = water.point_data["electron_density"]
print("density range:", float(density.min()), float(density.max()))
print("selected quantiles:", np.quantile(density, [0.99, 0.995, 0.999]))
water

In [ ]:
density_surfaces = water.contour(isosurfaces=[0.5, 1.0], scalars="electron_density")

def show_watermolecule_isosurface(level=0.5, opacity=0.72):
    levels = [level, min(level * 2.0, float(density.max()))]
    surfaces = water.contour(isosurfaces=levels, scalars="electron_density")
    p = pv.Plotter(window_size=(900, 650))
    p.add_mesh(
        surfaces,
        scalars="electron_density",
        cmap="viridis",
        opacity=opacity,
        smooth_shading=True,
        scalar_bar_args={"title": "electron density"},
    )
    p.add_mesh(water.outline(), color="black", line_width=1)
    p.add_text(f"Electron-density isosurfaces: {levels[0]:.2f}, {levels[1]:.2f}", position="upper_left", font_size=12)
    p.add_axes(line_width=2)
    p.camera_position = [(3.8, -7.0, 4.2), (1.0, 0.0, -1.0), (0, 0, 1)]
    return p.show()

widgets.interact(
    show_watermolecule_isosurface,
    level=widgets.FloatSlider(value=0.5, min=0.1, max=1.4, step=0.05, description="level", continuous_update=False),
    opacity=widgets.FloatSlider(value=0.72, min=0.2, max=1.0, step=0.05, description="opacity", continuous_update=False),
);

In [ ]:
water_png = OUTPUT_DIR / "case2_watermolecule_isosurface.png"
p = pv.Plotter(off_screen=True, window_size=(1200, 850))
p.add_mesh(density_surfaces, scalars="electron_density", cmap="viridis", opacity=0.72, smooth_shading=True)
p.add_mesh(water.outline(), color="black", line_width=1)
p.add_text("Water molecule electron density", position="upper_left", font_size=12)
p.add_axes(line_width=2)
p.camera_position = [(3.8, -7.0, 4.2), (1.0, 0.0, -1.0), (0, 0, 1)]
p.screenshot(water_png)
p.close()
water_png

## 3. 气象模拟：云水标量场 + 风场 glyph

数据文件：

- `cloudwater.dx`：云水标量场
- `wind.dx`：三维风场向量

这个案例展示同一网格上的标量场和向量场如何叠加。

In [ ]:
storm = load_storm_cloud_and_wind(DATA_DIR)
print("cloudwater range:", float(storm["cloudwater"].min()), float(storm["cloudwater"].max()))
print("wind speed range:", float(storm["wind_magnitude"].min()), float(storm["wind_magnitude"].max()))
storm

In [ ]:
cloud_surface = storm.contour(isosurfaces=[0.5], scalars="cloudwater")
sampled_wind = storm.extract_subset((0, 24, 0, 13, 0, 7), rate=(4, 3, 2))
wind_glyphs = sampled_wind.glyph(orient="wind", scale="wind_magnitude", factor=500.0)

def show_storm_cloud_wind(cloud_level=0.5, wind_stride=4, glyph_factor=500.0, cloud_opacity=0.55):
    cloud = storm.contour(isosurfaces=[cloud_level], scalars="cloudwater")
    rate = (wind_stride, max(1, wind_stride // 2), max(1, wind_stride // 2))
    sampled = storm.extract_subset((0, 24, 0, 13, 0, 7), rate=rate)
    glyphs = sampled.glyph(orient="wind", scale="wind_magnitude", factor=glyph_factor)
    p = pv.Plotter(window_size=(900, 650))
    p.add_mesh(
        cloud,
        scalars="cloudwater",
        cmap="Blues",
        opacity=cloud_opacity,
        smooth_shading=True,
        scalar_bar_args={"title": "cloudwater"},
    )
    p.add_mesh(glyphs, color="black")
    p.add_mesh(storm.outline(), color="gray", line_width=1)
    p.add_text(f"Cloudwater = {cloud_level:.2f}, wind stride = {wind_stride}", position="upper_left", font_size=12)
    p.add_axes(line_width=2)
    p.camera_position = [(130000, -90000, 70000), (50000, 14000, 16000), (0, 0, 1)]
    return p.show()

widgets.interact(
    show_storm_cloud_wind,
    cloud_level=widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1, description="cloud", continuous_update=False),
    wind_stride=widgets.IntSlider(value=4, min=2, max=6, step=1, description="stride", continuous_update=False),
    glyph_factor=widgets.FloatSlider(value=500.0, min=150.0, max=1200.0, step=50.0, description="arrow", continuous_update=False),
    cloud_opacity=widgets.FloatSlider(value=0.55, min=0.2, max=0.9, step=0.05, description="opacity", continuous_update=False),
);

In [ ]:
storm_png = OUTPUT_DIR / "case3_storm_cloud_wind.png"
p = pv.Plotter(off_screen=True, window_size=(1200, 850))
p.add_mesh(cloud_surface, scalars="cloudwater", cmap="Blues", opacity=0.55, smooth_shading=True)
p.add_mesh(wind_glyphs, color="black")
p.add_mesh(storm.outline(), color="gray", line_width=1)
p.add_text("Storm cloudwater + wind", position="upper_left", font_size=12)
p.add_axes(line_width=2)
p.camera_position = [(130000, -90000, 70000), (50000, 14000, 16000), (0, 0, 1)]
p.screenshot(storm_png)
p.close()
storm_png

## 4. MRI 医学体数据切片

数据文件：

- `MRI.data`：128 x 128 x 16 unsigned short 体数据
- `mri.general`：OpenDX general importer 头文件

这里先展示三个正交切片。体渲染可以进一步用 `add_volume` 和 opacity transfer function 扩展。

In [ ]:
mri = load_mri(DATA_DIR)
print("MRI intensity range:", int(mri["intensity"].min()), int(mri["intensity"].max()))
mri

In [ ]:
mri_slices = mri.slice_orthogonal()
mri_origin = np.array(mri.origin)
mri_spacing = np.array(mri.spacing)

def show_mri_slices(x_index=64, y_index=64, z_index=8, window_max=42000):
    origin = mri_origin + mri_spacing * np.array([x_index, y_index, z_index])
    slices = mri.slice_orthogonal(x=float(origin[0]), y=float(origin[1]), z=float(origin[2]))
    p = pv.Plotter(window_size=(900, 650))
    p.add_mesh(
        slices,
        scalars="intensity",
        cmap="gray",
        clim=(0, window_max),
        scalar_bar_args={"title": "intensity"},
    )
    p.add_mesh(mri.outline(), color="black", line_width=1)
    p.add_text(f"MRI slices: x={x_index}, y={y_index}, z={z_index}", position="upper_left", font_size=12)
    p.add_axes(line_width=2)
    p.camera_position = [(360, -430, 260), (110, 110, 18), (0, 0, 1)]
    return p.show()

widgets.interact(
    show_mri_slices,
    x_index=widgets.IntSlider(value=64, min=0, max=mri.dimensions[0] - 1, step=1, description="x", continuous_update=False),
    y_index=widgets.IntSlider(value=64, min=0, max=mri.dimensions[1] - 1, step=1, description="y", continuous_update=False),
    z_index=widgets.IntSlider(value=8, min=0, max=mri.dimensions[2] - 1, step=1, description="z", continuous_update=False),
    window_max=widgets.IntSlider(value=42000, min=5000, max=int(mri["intensity"].max()), step=1000, description="window", continuous_update=False),
);

In [ ]:
mri_png = OUTPUT_DIR / "case4_mri_slices.png"
p = pv.Plotter(off_screen=True, window_size=(1200, 850))
p.add_mesh(mri_slices, scalars="intensity", cmap="gray")
p.add_mesh(mri.outline(), color="black", line_width=1)
p.add_text("MRI orthogonal slices", position="upper_left", font_size=12)
p.add_axes(line_width=2)
p.camera_position = [(360, -430, 260), (110, 110, 18), (0, 0, 1)]
p.screenshot(mri_png)
p.close()
mri_png

## 5. 小结

这四个案例覆盖了三维科学可视化中最常见的四类任务：地形曲面、标量场等值面、标量/向量场叠加、医学体数据切片。后续计算模拟项目可以沿用同样的模式：把求解器输出转成 PyVista/VTK 数据对象，再选择合适的 filter 和渲染方式。